# EDA - Fraud_Data.csv (E-commerce Transactions)

## Objective
Comprehensive exploratory data analysis and preprocessing of e-commerce fraud data:
- Data cleaning (missing values, duplicates, data types)
- Univariate and bivariate analysis
- Class imbalance quantification
- Geolocation integration (IP-to-country mapping)
- Feature engineering (time features, transaction velocity)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded successfully')

## 1. Data Loading and Initial Exploration

In [ ]:
fraud_data = pd.read_csv('../data/raw/Fraud_Data.csv')
ip_country = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

print('Fraud Data shape:', fraud_data.shape)
display(fraud_data.head())
print(fraud_data.dtypes)
print('\nIP-to-Country shape:', ip_country.shape)
display(ip_country.head())

## 2. Data Cleaning

In [ ]:
missing = fraud_data.isnull().sum()
print('Missing values:', missing[missing>0] if missing.sum()>0 else 'None')
print(f'Total rows: {len(fraud_data)}')
print(f'Duplicates: {fraud_data.duplicated().sum()}')
print(f'Dup (user,device,time): {fraud_data.duplicated(subset=["user_id","device_id","purchase_time"]).sum()}')
fraud_clean = fraud_data.drop_duplicates().copy()
print(f'After dedup: {len(fraud_clean)} rows (removed {len(fraud_data)-len(fraud_clean)})')

In [ ]:
fraud_clean['signup_time'] = pd.to_datetime(fraud_clean['signup_time'])
fraud_clean['purchase_time'] = pd.to_datetime(fraud_clean['purchase_time'])
fraud_clean['sex'] = fraud_clean['sex'].astype('category')
fraud_clean['class'] = fraud_clean['class'].astype('int8')
# ip_address is already numeric float - cast to int for range lookups
fraud_clean['ip_address_int'] = fraud_clean['ip_address'].round().astype('Int64')
print(fraud_clean.dtypes)

## 3. Univariate Analysis

In [ ]:
print(fraud_clean.describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].hist(fraud_clean['age'], bins=30, edgecolor='black', alpha=0.7)
axes[0,0].set_title('Distribution of Age', fontweight='bold')
axes[0,1].hist(fraud_clean['purchase_value'], bins=50, edgecolor='black', alpha=0.7)
axes[0,1].set_title('Distribution of Purchase Value', fontweight='bold')
axes[1,0].hist(np.log1p(fraud_clean['purchase_value']), bins=50, edgecolor='black', alpha=0.7)
axes[1,0].set_title('Log(Purchase Value)', fontweight='bold')
fraud_clean['sex'].value_counts().plot(kind='bar', ax=axes[1,1], edgecolor='black')
axes[1,1].set_title('Gender Distribution', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/univariate_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f"Users: {fraud_clean['user_id'].nunique():,}")
print(f"Devices: {fraud_clean['device_id'].nunique():,}")
print(f"Browsers: {fraud_clean['browser'].nunique()}")
print(f"Sources: {fraud_clean['source'].nunique()}")
print('\nBrowser distribution:')
print(fraud_clean['browser'].value_counts().head(10))
print('\nSource distribution:')
print(fraud_clean['source'].value_counts())

## 4. Class Imbalance Analysis

In [ ]:
cc = fraud_clean['class'].value_counts()
cp = fraud_clean['class'].value_counts(normalize=True)*100
print(f'Legitimate (0): {cc[0]:,} ({cp[0]:.2f}%)')
print(f'Fraudulent (1): {cc[1]:,} ({cp[1]:.2f}%)')
print(f'Ratio: {cc[0]/cc[1]:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cc.plot(kind='bar', ax=axes[0], color=['green','red'], edgecolor='black')
axes[0].set_title('Class Distribution (Count)', fontweight='bold')
axes[0].set_xticklabels(['Legitimate','Fraudulent'], rotation=0)
cp.plot(kind='pie', ax=axes[1], labels=['Legitimate','Fraudulent'],
      autopct='%1.2f%%', colors=['green','red'])
axes[1].set_title('Class Distribution (%)', fontweight='bold')
axes[1].set_ylabel('')
plt.tight_layout()
plt.savefig('../data/processed/class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fraud_clean.boxplot(column='purchase_value', by='class', ax=axes[0,0])
axes[0,0].set_title('Purchase Value by Class', fontweight='bold')
fraud_clean.boxplot(column='age', by='class', ax=axes[0,1])
axes[0,1].set_title('Age by Class', fontweight='bold')
bct = pd.crosstab(fraud_clean['browser'], fraud_clean['class'])
bfr = bct[1]/bct.sum(axis=1)*100
bfr.sort_values(ascending=False).head(10).plot(kind='barh', ax=axes[1,0], edgecolor='black')
axes[1,0].set_title('Fraud Rate by Browser (Top 10)', fontweight='bold')
axes[1,0].set_xlabel('Fraud Rate (%)')
sct = pd.crosstab(fraud_clean['source'], fraud_clean['class'])
sfr = sct[1]/sct.sum(axis=1)*100
sfr.plot(kind='bar', ax=axes[1,1], edgecolor='black')
axes[1,1].set_title('Fraud Rate by Source', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../data/processed/bivariate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Geolocation Integration

**Note:** `ip_address` in Fraud_Data.csv is already numeric (float), not dotted-quad. We round and cast to int for the range-based merge with IpAddress_to_Country.csv.

In [ ]:
fraud_valid = fraud_clean.dropna(subset=['ip_address_int']).copy()
fraud_valid['ip_address_int'] = fraud_valid['ip_address_int'].astype('int64')
print(f'Valid IPs: {len(fraud_valid):,}')

fs = fraud_valid.sort_values('ip_address_int').reset_index(drop=True)
ips = ip_country.copy()
ips['lower_bound_ip_address'] = ips['lower_bound_ip_address'].astype('int64')
ips['upper_bound_ip_address'] = ips['upper_bound_ip_address'].astype('int64')
ips = ips.sort_values('lower_bound_ip_address').reset_index(drop=True)

fraud_geo = pd.merge_asof(fs, ips[['lower_bound_ip_address','upper_bound_ip_address','country']],
    left_on='ip_address_int', right_on='lower_bound_ip_address', direction='backward')
mask = fraud_geo['ip_address_int'] <= fraud_geo['upper_bound_ip_address']
fraud_geo.loc[~mask | fraud_geo['country'].isna(), 'country'] = 'Unknown'
print(f"Matched: {(fraud_geo['country']!='Unknown').sum():,}")
print(f"Unmatched: {(fraud_geo['country']=='Unknown').sum():,}")
print(fraud_geo['country'].value_counts().head(10))


In [ ]:
cf = (fraud_geo[fraud_geo['country']!='Unknown'].groupby('country')['class']
      .agg(['sum','count','mean']))
cf.columns = ['fraud_count','total_txn','fraud_rate']
cf['fraud_rate'] *= 100
cf = cf.sort_values('total_txn', ascending=False)
print(cf.head(15))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
t15 = cf.head(15)
t15['fraud_rate'].plot(kind='barh', ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('Fraud Rate - Top 15 Countries', fontweight='bold')
t15['total_txn'].plot(kind='barh', ax=axes[1], color='skyblue', edgecolor='black')
axes[1].set_title('Volume - Top 15 Countries', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/fraud_by_country.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Time-Based Feature Engineering

In [ ]:
fraud_geo['time_since_signup'] = (
    (fraud_geo['purchase_time']-fraud_geo['signup_time']).dt.total_seconds()/3600).clip(lower=0)
fraud_geo['hour_of_day'] = fraud_geo['purchase_time'].dt.hour
fraud_geo['day_of_week'] = fraud_geo['purchase_time'].dt.dayofweek
fraud_geo['day_name'] = fraud_geo['purchase_time'].dt.day_name()
fraud_geo['month'] = fraud_geo['purchase_time'].dt.month
print('Time since signup (hours):')
print(fraud_geo['time_since_signup'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(fraud_geo['time_since_signup'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Time Since Signup (hours)', fontweight='bold')
axes[0].set_xlim(0, 1000)
fraud_geo.boxplot(column='time_since_signup', by='class', ax=axes[1])
axes[1].set_title('Time Since Signup by Class', fontweight='bold')
axes[1].set_ylim(0, 1000)
plt.tight_layout()
plt.savefig('../data/processed/time_since_signup.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Function to convert IP address to integer
def ip_to_int(ip_val):
    """Convert IP address (either numeric or dotted string) to integer, handle NaN values"""
    if pd.isna(ip_val):
        return np.nan
    if isinstance(ip_val, (int, float, np.integer, np.floating)):
        return int(round(ip_val))
    try:
        ip_str = str(ip_val).strip()
        if '.' in ip_str:
            parts = ip_str.split('.')
            if len(parts) == 4:
                return (int(parts[0]) << 24) + (int(parts[1]) << 16) + (int(parts[2]) << 8) + int(parts[3])
        return int(float(ip_str))
    except (ValueError, AttributeError):
        return np.nan

# Test the function
test_ip = fraud_clean['ip_address'].iloc[0]
print(f"Test IP: {test_ip}")
print(f"IP as integer: {ip_to_int(test_ip)}")
print(f"Test with NaN: {ip_to_int(np.nan)}")


## 8. Transaction Velocity Features

In [ ]:
utc = fraud_geo.groupby('user_id').size().reset_index(name='user_transaction_count')
fraud_geo = fraud_geo.merge(utc, on='user_id')
print(fraud_geo['user_transaction_count'].describe())

# 24h rolling count (vectorized per user group)
fraud_geo = fraud_geo.sort_values(['user_id','purchase_time']).reset_index(drop=True)
def count_24h(s):
    return s.apply(lambda t: ((s>=t-pd.Timedelta(hours=24))&(s<=t)).sum())
fraud_geo['txn_in_24h'] = fraud_geo.groupby('user_id')['purchase_time'].transform(count_24h)
print('\n24h txn frequency:')
print(fraud_geo['txn_in_24h'].describe())

In [ ]:
# Redundant lookup cell cleared to avoid duplication


## 9. Summary Report

In [ ]:
n_legit = (fraud_geo['class']==0).sum()
n_fraud = (fraud_geo['class']==1).sum()
print(f'Records: {len(fraud_geo):,}')
print(f'Legitimate: {n_legit:,} ({n_legit/len(fraud_geo)*100:.2f}%)')
print(f'Fraudulent: {n_fraud:,} ({n_fraud/len(fraud_geo)*100:.2f}%)')
print(f'Ratio: {n_legit/n_fraud:.1f}:1')
print(f"Countries matched: {(fraud_geo['country']!='Unknown').nunique()}")
print(f"Mean time_since_signup: {fraud_geo['time_since_signup'].mean():.1f} hrs")

report = f'''{'='*60}\nEDA SUMMARY - Fraud_Data.csv\n{'='*60}\n"
Records: {len(fraud_geo):,}\n"
Class Imbalance: {n_legit/len(fraud_geo)*100:.2f}% legit / {n_fraud/len(fraud_geo)*100:.2f}% fraud\n"
Geolocation matched: {(fraud_geo['country']!='Unknown').sum():,}\n"
{'='*60}'''
with open('../data/processed/EDA_Summary_Report.txt','w') as f: f.write(report)
print(report)

## 10. Save Processed Data

In [ ]:
save_cols = [c for c in fraud_geo.columns
             if c not in ('ip_address_int','lower_bound_ip_address','upper_bound_ip_address')]
fraud_geo[save_cols].to_csv('../data/processed/fraud_data_cleaned_eda.csv', index=False)
print(f'Saved fraud_data_cleaned_eda.csv ({len(fraud_geo):,} rows)')
print('Done!')